# Od liczby dziesiętnej do bitów — i z powrotem
### W2 · Materiał pomocniczy do *Podstaw programowania*

**Cel:** samodzielnie przeliczyć kilka liczb, zobaczyć skąd biorą się bity części ułamkowej i dopiero potem odczytać znaczenie pól `float`.

**Nie trzeba liczyć w głowie ani znać kodów na pamięć.** Wystarczy umieć powtórzyć procedurę na kartce. Przykłady są przygotowane jako krótkie doświadczenia: **przewidź → policz → sprawdź**.

Notebook jest uzupełnieniem W2 (a nie nowym wykładem). Komórki wykonawcze są w **C11**, zgodnie z kernelem C używanym na zajęciach. Gdy pojawiają się pętle, można potraktować je jak gotowy kalkulator — składnię pętli poznamy później.

| Co będzie po kolei? | Cel |
|---|---|
| 1. Liczby całkowite: `10₂` i inne | Wagi bitów oraz dzielenie przez 2 |
| 2. Ułamki binarne | Wagi `1/2`, `1/4`, `1/8`, … oraz mnożenie przez 2 |
| 3. `0.1` i dokładność | Dlaczego rozwinięcie może się nie kończyć |
| 4. `0.123687` i `1.123687` | Różnica między jedynką przed przecinkiem a **ukrytą jedynką** IEEE 754 |
| 5. Krótka powtórka U2 | Dlaczego `10000101₂` nie oznacza `−5` w U2 |

## 1 · Zapis pozycyjny: dlaczego `10₂ = 2₁₀`?

Podstawa systemu informuje, jakie są **wagi pozycji**. W systemie dziesiętnym mamy `1`, `10`, `100`, …, a w binarnym `1`, `2`, `4`, `8`, `16`, `32`, …

`(10)₂ = 1 · 2¹ + 0 · 2⁰ = (2)₁₀`.

Aby przejść **z systemu dziesiętnego do binarnego**, możemy dzielić liczbę przez 2, zapisywać reszty i odczytać je **od ostatniej do pierwszej**.

Dla **13**:

| Działanie | Iloraz | Reszta |
|:---|---:|---:|
| 13 ÷ 2 | 6 | **1** |
| 6 ÷ 2 | 3 | **0** |
| 3 ÷ 2 | 1 | **1** |
| 1 ÷ 2 | 0 | **1** |

Odczyt od dołu: `1101₂`. Weryfikacja w drugą stronę: `8 + 4 + 0 + 1 = 13`.

| Dziesiętnie | Rozkład na potęgi 2 | Binarnie | Sprawdzenie w drugą stronę |
|---:|:---|:---|:---|
| **2** | 2 | `10₂` | 2 |
| **13** | 8 + 4 + 1 | `1101₂` | 8 + 4 + 1 = 13 |
| **24** | 16 + 8 | `11000₂` | 16 + 8 = 24 |
| **32** | 32 | `100000₂` | 32 |

**Pytanie:** dlaczego zapis `100000₂` odpowiada dokładnie 32, chociaż zawiera pięć zer?

### Sprawdź programem C (opcjonalnie)

Kod wypisuje bity odpowiednio do wag `32, 16, 8, 4, 2, 1`. **Nie trzeba teraz analizować składni pętli** — skup się na otrzymanym zapisie.

Przewidywany wynik:

```text
2_10 = 10_2
13_10 = 1101_2
24_10 = 11000_2
32_10 = 100000_2
```

In [ ]:
#include <stdio.h>

void pokaz_bity(unsigned int liczba)
{
    unsigned int wagi[] = {32u, 16u, 8u, 4u, 2u, 1u};
    int zaczete = 0;
    for (int i = 0; i < 6; ++i) {
        unsigned int bit = (liczba / wagi[i]) % 2u;
        if (bit != 0u) zaczete = 1;
        if (zaczete) putchar((int)('0' + bit));
    }
    if (!zaczete) putchar('0');
}

int main(void)
{
    unsigned int liczby[] = {2u, 13u, 24u, 32u};
    for (int i = 0; i < 4; ++i) {
        printf("%u_10 = ", liczby[i]);
        pokaz_bity(liczby[i]);
        puts("_2");
    }
    return 0;
}

## 2 · Ułamki: co oznaczają bity po przecinku?

Zasada pozycyjna działa również **po przecinku**, tylko wagi są ułamkami potęg dwójki:

| Pozycja | Pierwsza | Druga | Trzecia | Czwarta | Piąta |
|---|---:|---:|---:|---:|---:|
| Waga | `2⁻¹ = 1/2` | `2⁻² = 1/4` | `2⁻³ = 1/8` | `2⁻⁴ = 1/16` | `2⁻⁵ = 1/32` |

Dlatego:

`(0.101)₂ = 1/2 + 0/4 + 1/8 = (0.625)₁₀`.

**Jak przejść z dziesiętnej części ułamkowej do binarnej?** Pomnóż resztę przez 2. **Część całkowita** wyniku jest następnym bitem, a pozostałą część ułamkową mnożysz ponownie.

| Krok | Mnożenie przez 2 | Następny bit | Nowa reszta |
|---:|:---|:---:|:---|
| 1 | `0.625 × 2 = 1.25` | **1** | `0.25` |
| 2 | `0.25 × 2 = 0.5` | **0** | `0.5` |
| 3 | `0.5 × 2 = 1.0` | **1** | `0` |

Zapisujemy bity w kolejności: `0.101₂`. Reszta wyniosła zero, więc rozwinięcie się skończyło.

**Pytanie:** jak zapisać `0.375₁₀`? Sprawdź: `0.375 × 2 = 0.75` → bit 0; `0.75 × 2 = 1.5` → bit 1; `0.5 × 2 = 1.0` → bit 1. Wynik: `0.011₂`.

## 3 · Dlaczego `0.1₁₀` nie kończy się w systemie binarnym?

Spróbujmy tej samej procedury dla dokładnego ułamka `1/10`. Tutaj wygodnie posługiwać się **resztą jako ułamkiem**, żeby nie zaokrąglać niczego po drodze.

| Krok | Reszta przed mnożeniem | Po pomnożeniu przez 2 | Bit | Nowa reszta |
|---:|---:|---:|:---:|---:|
| 1 | `0.1` | `0.2` | **0** | `0.2` |
| 2 | `0.2` | `0.4` | **0** | `0.4` |
| 3 | `0.4` | `0.8` | **0** | `0.8` |
| 4 | `0.8` | `1.6` | **1** | `0.6` |
| 5 | `0.6` | `1.2` | **1** | `0.2` |

Widzimy tę samą resztę `0.2` co po pierwszym kroku. Dalej będą powtarzać się bity `0011`:

**`(0.1)₁₀ = (0.00011001100110011...)₂`**.

Ważna reguła: **ułamek wymierny ma skończone rozwinięcie binarne wtedy i tylko wtedy, gdy po skróceniu jego mianownik jest potęgą 2**.

- `0.5 = 1/2 = 0.1₂` — **dokładnie**.
- `0.625 = 5/8 = 0.101₂` — **dokładnie**.
- `2.5 = 10.1₂` — **dokładnie**.
- `0.1 = 1/10` — **nieskończenie wiele bitów**, więc `float`/`double` przechowują zaokrąglone przybliżenie.
- `2.1 = 2 + 1/10` — również ma **nieskończone rozwinięcie binarne**.

**Pytanie:** czy większy typ `double` sprawia, że rozwinięcie `0.1` staje się skończone? **Nie** — pozwala jedynie zapisać więcej jego bitów.

### Kalkulator rozwinięcia binarnego — bez błędów `float`

Ten kod oblicza bity dokładnych ułamków `licznik/mianownik`, korzystając wyłącznie z **arytmetyki całkowitej**. Dzięki temu nie wprowadza błędu `float` już podczas konwersji.

Dla `1/10` zobaczysz powtarzające się `0011`; dla `123687/1000000` ułamek binarny również nie zakończy się po kilkunastu bitach.

In [ ]:
#include <stdio.h>

void rozwin_ulamek(const char *nazwa, unsigned int calkowita,
                   unsigned int licznik, unsigned int mianownik,
                   unsigned int ile_bitow)
{
    unsigned int reszta = licznik;
    printf("%s = %u.", nazwa, calkowita);
    for (unsigned int i = 0; i < ile_bitow && reszta != 0u; ++i) {
        reszta *= 2u;
        unsigned int bit = reszta / mianownik;
        reszta %= mianownik;
        putchar((int)('0' + bit));
    }
    putchar('\n');
}

int main(void)
{
    rozwin_ulamek("0.625", 0u, 5u, 8u, 20u);
    rozwin_ulamek("0.1", 0u, 1u, 10u, 24u);
    rozwin_ulamek("0.123687", 0u, 123687u, 1000000u, 32u);
    rozwin_ulamek("1.123687", 1u, 123687u, 1000000u, 32u);
    return 0;
}

## 4 · Dwie liczby: `0.123687` i `1.123687`

Zapis dziesiętny tych liczb różni się o **dokładnie 1**. Zatem ich **matematyczne, nieskończone rozwinięcia binarne** mają te same bity części ułamkowej:

```text
0.123687₁₀  =  0.000111111010100111110011...₂
1.123687₁₀  =  1.000111111010100111110011...₂
```

**Skąd wzięła się jedynka przed przecinkiem?** To zwykła część całkowita liczby `1.123687`; przy `0.123687` część całkowita wynosi 0. **To jeszcze nie jest „ukryta jedynka” formatu `float`.**

### Teraz dopiero normalizacja

Dla dodatniej **liczby znormalizowanej** zapis ma postać `(1.F)₂ × 2ᵉ`. Przesuwamy kropkę binarną tak, aby po lewej została jedna jedynka:

| Liczba | Postać znormalizowana (początek) | Wykładnik `e` |
|---|---|---:|
| `0.123687` | `1.11111010100111110011…₂ × 2⁻⁴` | `−4` |
| `1.123687` | `1.00011111101010011111…₂ × 2⁰` | `0` |

**Uwaga:** w obu przypadkach znacząca zaczyna się od **`1.`**, mimo że pierwsza liczba jest mniejsza od jedności. To właśnie **jedynka domyślna (ukryta)**, której nie zapisujemy w polu `F` znormalizowanych liczb IEEE 754.

**Kiedy jej nie ma?** W binary32, gdy **`E = 0`**: dla liczb podnormalnych znacząca zaczyna się od `0.F`, a przy `E=0`, `F=0` mamy zero. To osobna reguła interpretacji wzorca bitowego, **nie decyzja oparta na części całkowitej liczby dziesiętnej**.

### Zapis dwóch liczb w IEEE 754 binary32 (`float`)

W typowej konfiguracji Linux/x86-64 i GCC używamy 32-bitowego formatu `float`: **1 bit znaku `S`, 8 bitów wykładnika `E`, 23 bity pola ułamkowego `F`**.

Dla liczb znormalizowanych: `wartość = (−1)^S × (1.F)₂ × 2^(E−127)`.

| Wartość z kodu C | `S` | `E` — 8 bitów | `e=E−127` | `F` — 23 bity (już po zaokrągleniu) |
|---|:---:|:---|---:|:---|
| `0.123687f` | `0` | `01111011` (123) | `−4` | `11111010100111110011100` |
| `1.123687f` | `0` | `01111111` (127) | `0` | `00011111101010011111010` |

**Dlaczego zapisane pola `F` są różne, skoro dziesiętne części ułamkowe są takie same?** Normalizacja przesuwa kropkę binarną w inne miejsce; następnie pole `F` przechowuje tylko **23 kolejne bity** i zaokrągla dalsze. To może prowadzić do różnych błędów przybliżenia.

Zobaczmy wartości faktycznie zapisane w `float` (na konfiguracji referencyjnej):

- `0.123687f ≈ 0.12368699908256531`;
- `1.123687f ≈ 1.1236870288848877`.

Do odczytu bitów w C używamy `memcpy` (bez niepoprawnego rzutowania wskaźników). **Operacje bitowe w kodzie można na razie traktować jako gotowe narzędzie diagnostyczne**.

In [ ]:
#include <stdio.h>
#include <stdint.h>
#include <string.h>
#include <float.h>

void pokaz_float(const char *etykieta, float liczba)
{
    uint32_t bity = 0u;
    memcpy(&bity, &liczba, sizeof(liczba));
    unsigned int znak = bity >> 31u;
    unsigned int E = (bity >> 23u) & 255u;
    unsigned int F = bity & 0x7fffffu;

    printf("%-11s : %u | ", etykieta, znak);
    for (int i = 7; i >= 0; --i) putchar('0' + ((E >> i) & 1u));
    printf(" | ");
    for (int i = 22; i >= 0; --i) putchar('0' + ((F >> i) & 1u));
    printf(" ; E=%u, e=%d\n", E, (int)E - 127);
}

int main(void)
{
    if (sizeof(float) != 4 || sizeof(uint32_t) != 4 ||
        FLT_RADIX != 2 || FLT_MANT_DIG != 24) {
        puts("To cwiczenie wymaga formatu IEEE 754 binary32.");
        return 0;
    }
    pokaz_float("0.123687f", 0.123687f);
    pokaz_float("1.123687f", 1.123687f);
    return 0;
}

## 5 · A kiedy ukrytej jedynki naprawdę nie ma?

Nie decyduje o tym, czy zapis dziesiętny zaczyna się od `0` czy od `1`. Decyduje pole wykładnika **`E`**.

| Pole `E` w binary32 | Pole `F` | Co oznacza? |
|---|---|---|
| `00000000` | same zera | **Zero** (`+0` albo `−0`) |
| `00000000` | przynajmniej jedna jedynka | **Liczba podnormalna**: znacząca `0.F`, wykładnik `−126` |
| `00000001`…`11111110` | dowolne | **Liczba znormalizowana**: znacząca `1.F`, wykładnik `E−127` |
| `11111111` | same zera | Nieskończoność |
| `11111111` | inne niż zero | NaN |

Porównaj kilka **sąsiednich** dodatnich wartości:

- najmniejsza podnormalna: `E=0`, `F=00…001`, wartość `2⁻¹⁴⁹`;
- druga podnormalna: `E=0`, `F=00…010`, wartość `2⁻¹⁴⁸`;
- największa podnormalna: `E=0`, `F=11…111`, wartość `2⁻¹²⁶ − 2⁻¹⁴⁹`;
- najmniejsza znormalizowana: `E=1`, `F=00…000`, wartość `2⁻¹²⁶`.

**Pytanie:** przy zmianie z największej podnormalnej na najmniejszą znormalizowaną ukryta jedynka „pojawia się” w interpretacji, ale czy pojawia się duża przerwa między reprezentowalnymi liczbami? **Nie**: odstęp wynosi `2⁻¹⁴⁹`.

## 6 · Powtórka z U2: pierwsza jedynka ma inne znaczenie

To niezależne zagadnienie od wykładnika i ukrytej jedynki w `float`.

W 8-bitowym zapisie `10000101` otrzymujemy:

| Interpretacja | Obliczenie | Wynik |
|---|---|---:|
| Bez znaku (`unsigned`) | `128 + 4 + 1` | **133** |
| U2 (uzupełnienie do dwóch) | `−128 + 4 + 1` | **−123** |
| Znak–moduł | „minus” i `0000101` | **−5** |

**Najczęstsza pułapka:** w U2 najstarszy bit ma wagę **ujemną**, zamiast być osobną etykietą znaku. To dlatego `10000101` w U2 **nie oznacza** `−5`.

## 7 · Sprawdź się (bez presji)

1. Zamień `(101.011)₂` na system dziesiętny.
2. Zamień `(0.375)₁₀` na system binarny.
3. Czy `0.25`, `0.2`, `5.625` mają dokładne skończone rozwinięcie binarne? Dlaczego?
4. Jakie pola `E` (dziesiętnie) mają znormalizowane liczby `0.123687f` i `1.123687f` w binary32?
5. Czy przy `E=0` i `F≠0` mamy wiodącą `1` w znaczącej?
6. Jaką wartość ma `10000101₂` jako 8-bitowa liczba U2, a jaką jako bez znaku?

<details>
<summary><b>Pokaż odpowiedzi</b></summary>

1. `5 + 1/4 + 1/8 = 5.375`.
2. `0.011₂`.
3. `0.25 = 1/4 = 0.01₂` i `5.625 = 5 + 5/8 = 101.101₂` są dokładne; `0.2 = 1/5` ma nieskończone rozwinięcie binarne.
4. Dla `0.123687f`: `e=−4`, więc `E=123`. Dla `1.123687f`: `e=0`, więc `E=127`.
5. Nie. Podnormalna ma znaczącą `0.F` i rzeczywisty wykładnik `−126`.
6. U2: `−123`; unsigned: `133`.

</details>

**Wskazówka:** gdyby trzeba było zapamiętać tylko jedną regułę, niech będzie nią *„rozpisz liczbę na wagi pozycji i dopiero potem interpretuj bity zgodnie z wybranym typem”*. 

**Dalszy krok:** materiał w W2 o `float` i `double` oraz ściąga A4 do tego notebooka.